In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

orders_df = pd.DataFrame({
    'shop': ['Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town'],
    'weekday': ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
    'croissants': [99, 79, 89, 80, 73, 112, 112, 81, 66, 91, 67, 75, 126, 118, 95, 71, 85, 90, 97, 111, 123, 79, 98, 87, 92, 78, 98, 135, 81, 86, 85, 86, 93, 122, 116, 56, 89, 68, 77, 71, 117, 121],
    'loaves': [42, 53, 45, 43, 48, 60, 39, 44, 55, 32, 53, 54, 66, 57, 40, 58, 53, 52, 53, 60, 50, 44, 28, 57, 54, 50, 75, 63, 48, 51, 38, 49, 56, 52, 68, 50, 56, 49, 52, 35, 52, 52],
    'cakes': [22, 7, 13, 21, 15, 32, 35, 33, 12, 14, 11, 20, 31, 32, 35, 19, 21, 19, 22, 28, 33, 27, 24, 10, 12, 19, 26, 30, 26, 19, 22, 30, 22, 32, 37, 15, 26, 15, 10, 18, 25, 37],
    'temperature_c': [11.4, 4.9, 6.8, 23.7, 8.2, 19.6, 22.3, 26.5, 4.9, 17.3, 5.4, 4.5, 7.0, 22.8, 26.0, 4.1, 15.9, 13.0, 8.8, 4.6, 12.4, 11.9, 16.1, 10.0, 11.6, 9.2, 10.0, 9.1, 23.3, 21.3, 26.3, 24.8, 12.1, 14.0, 9.5, 17.4, 16.0, 20.7, 6.4, 25.0, 14.4, 16.9],
    'rain_mm': [4.0, 2.8, 2.2, 1.4, 0.0, 2.2, 0.8, 0.0, 0.0, 0.0, 0.2, 4.4, 1.5, 0.0, 2.6, 0.3, 0.0, 0.0, 2.9, 3.2, 5.7, 2.2, 3.8, 0.0, 2.4, 0.0, 0.0, 0.7, 0.0, 0.0, 0.0, 2.7, 0.0, 0.3, 0.0, 4.1, 4.7, 4.5, 0.0, 0.0, 0.0, 0.7],
    'promotion': [True, False, False, False, False, False, False, True, False, False, False, False, False, False, True, False, True, False, True, False, False, False, True, False, False, False, False, True, False, False, False, False, True, False, False, False, True, False, False, False, False, False],
    'complaints': [2, 2, 1, 3, 1, 1, 1, 2, 0, 1, 0, 1, 0, 2, 0, 0, 2, 1, 2, 2, 1, 1, 2, 0, 1, 1, 2, 0, 2, 2, 1, 1, 0, 2, 0, 1, 1, 0, 0, 1, 1, 2],
})

In [ ]:
orders_df["total_items"] = orders_df[["croissants", "loaves", "cakes"]].sum(axis=1)
orders_df["is_weekend"] = orders_df["weekday"].isin(["Sat", "Sun"])
orders_df["cake_share"] = orders_df["cakes"] / orders_df["total_items"]
print(orders_df[["total_items", "cake_share"]].describe().round(3))

In [ ]:
resample_rng = np.random.default_rng(31)
bootstrap_rounds = 2000
confidence_level = 0.95

In [ ]:
def bootstrap_interval(values, statistic, rounds, level, rng):
    samples = [statistic(rng.choice(values, size=len(values), replace=True)) for _ in range(rounds)]
    lower = np.quantile(samples, (1 - level) / 2)
    upper = np.quantile(samples, 1 - (1 - level) / 2)
    return float(statistic(values)), float(lower), float(upper)


interval_rows = []
for shop_name, shop_rows in orders_df.groupby("shop"):
    estimate, lower, upper = bootstrap_interval(shop_rows["total_items"].to_numpy(), np.mean, bootstrap_rounds, confidence_level, resample_rng)
    interval_rows.append({"shop": shop_name, "metric": "mean items", "estimate": round(estimate, 2), "lower": round(lower, 2), "upper": round(upper, 2)})
    estimate, lower, upper = bootstrap_interval(shop_rows["cake_share"].to_numpy(), np.median, bootstrap_rounds, confidence_level, resample_rng)
    interval_rows.append({"shop": shop_name, "metric": "median cake share", "estimate": round(estimate, 4), "lower": round(lower, 4), "upper": round(upper, 4)})
interval_df = pd.DataFrame(interval_rows)
print(interval_df)

In [ ]:
weekend_values = orders_df.loc[orders_df["is_weekend"], "total_items"].to_numpy()
weekday_values = orders_df.loc[~orders_df["is_weekend"], "total_items"].to_numpy()
difference_samples = []
for _ in range(bootstrap_rounds):
    weekend_sample = resample_rng.choice(weekend_values, size=len(weekend_values), replace=True)
    weekday_sample = resample_rng.choice(weekday_values, size=len(weekday_values), replace=True)
    difference_samples.append(weekend_sample.mean() - weekday_sample.mean())
difference_samples = np.array(difference_samples)
difference_lower, difference_upper = np.quantile(difference_samples, [0.025, 0.975])
observed_difference = weekend_values.mean() - weekday_values.mean()
share_above_zero = float((difference_samples > 0).mean())
weekend_summary = {"observed": round(float(observed_difference), 2), "lower": round(float(difference_lower), 2), "upper": round(float(difference_upper), 2), "share_above_zero": round(share_above_zero, 3)}

In [ ]:
promotion_values = orders_df.loc[orders_df["promotion"], "croissants"].to_numpy()
regular_values = orders_df.loc[~orders_df["promotion"], "croissants"].to_numpy()
pooled_values = np.concatenate([promotion_values, regular_values])
observed_gap = promotion_values.mean() - regular_values.mean()
permutation_gaps = []
for _ in range(bootstrap_rounds):
    shuffled = resample_rng.permutation(pooled_values)
    permutation_gaps.append(shuffled[:len(promotion_values)].mean() - shuffled[len(promotion_values):].mean())
permutation_p_value = float((np.abs(permutation_gaps) >= abs(observed_gap)).mean())

In [ ]:
rain_bins = pd.cut(orders_df["rain_mm"], bins=[-0.1, 0.5, 3, 100], labels=["dry", "light", "wet"])
rain_effect_df = orders_df.groupby(rain_bins, observed=True)["croissants"].agg(["count", "mean", "std"]).round(2)

In [ ]:
temperature_bins = pd.cut(orders_df["temperature_c"], bins=[0, 10, 18, 30], labels=["cold", "mild", "warm"])
cake_by_temperature_df = orders_df.groupby(temperature_bins, observed=True)["cakes"].mean().round(2)

In [ ]:
complaint_rate = orders_df["complaints"].sum() / orders_df["total_items"].sum() * 1000

In [ ]:
print("Weekend minus weekday items:", weekend_summary)
for key_name in ("observed", "lower", "upper"):
    print(f"  {key_name}: {weekend_summary[key_name]}")
print(f"Promotion croissant gap {observed_gap:.2f} with permutation p-value {permutation_p_value:.4f}")

In [ ]:
print(rain_effect_df)
print(cake_by_temperature_df)
print(f"Complaints per thousand items: {complaint_rate:.3f}")

In [ ]:
shop_totals = orders_df.groupby("shop")["total_items"].sum()
shop_share = (shop_totals / shop_totals.sum()).round(4)
top_shop = shop_share.idxmax()
top_share = shop_share.max()
concentration_index = float((shop_share ** 2).sum())
print(f"Top shop {top_shop} with share {top_share:.2%}; concentration index {concentration_index:.4f}")

In [ ]:
weekly_positions = np.arange(len(orders_df)) // 7
orders_df["week_number"] = weekly_positions + 1
weekly_totals = orders_df.groupby("week_number")["total_items"].sum()
weekly_change = weekly_totals.pct_change().round(4)
for week_number, weekly_total in weekly_totals.items():
    change = weekly_change.loc[week_number]
    change_text = "n/a" if pd.isna(change) else f"{change:+.1%}"
    print(f"week {week_number}: {weekly_total} items, change {change_text}")

In [ ]:
print("Days with more than two complaints:", int((orders_df["complaints"] > 2).sum()))

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(difference_samples, bins=40)
axes[0].axvline(observed_difference, color="black")
axes[0].set_title("Weekend uplift bootstrap")
axes[1].bar(shop_share.index, shop_share.values)
axes[1].set_title("Share of items by shop")
plt.tight_layout()
plt.show()
print("Bootstrap figure drawn; intervals computed for", len(interval_df), "shop metrics")
for row in interval_df.itertuples(index=False):
    print(f"{row.shop:10s} {row.metric:18s} {row.estimate:>8} [{row.lower}, {row.upper}]")
print("Largest interval width:", round(float((interval_df["upper"] - interval_df["lower"]).max()), 4))
print("Smallest interval width:", round(float((interval_df["upper"] - interval_df["lower"]).min()), 4))
print("Weekend uplift share above zero:", weekend_summary["share_above_zero"])
print("Analysis complete.")